# **1. IMPORT LIBRARIES**

In [1]:
import pandas as pd
import numpy as np

# **2. LOAD DATA**

In [2]:
# load IMDb data
titles = pd.read_csv("title.basics.tsv", sep='\t')
ratings = pd.read_csv("title.ratings.tsv", sep='\t')
akas = pd.read_csv("title.akas.tsv", sep='\t')
crew = pd.read_csv("title.crew.tsv", sep='\t')
name_basic = pd.read_csv("name.basics.tsv", sep='\t')
principals = pd.read_csv("title.principals.tsv", sep='\t')


In [ ]:
titles['titleType'].unique()

# **3. DATA PROCESSING**

## **MOVIES**

In [3]:
movies = titles.query("titleType == 'movie'").copy()

# clean numeric fields
movies['startYear'] = pd.to_numeric(movies['startYear'], errors='coerce')
movies['runtimeMinutes'] = pd.to_numeric(movies['runtimeMinutes'], errors='coerce')
movies['isAdult'] = pd.to_numeric(movies['isAdult'], errors='coerce').fillna(0).astype(int)

# optional: remove adult films
movies = movies[movies['isAdult'] == 0]

# merge rating
movies = movies.merge(
    ratings[['tconst', 'averageRating', 'numVotes']],
    on='tconst', how='left'
)

movies['has_rating'] = movies['averageRating'].notna()
movies['is_new'] = ~movies['has_rating']

# genre processing
movies['genres'] = movies['genres'].fillna('Unknown')
movies['genre_list'] = movies['genres'].str.split(',')
movies['genre_count'] = movies['genre_list'].apply(lambda x: len([g for g in x if isinstance(g, str)]))

In [4]:
movies

,tconst,titleType,primaryTitle,originalTitle,isAdult,startYear,endYear,runtimeMinutes,genres,averageRating,numVotes,has_rating,is_new,genre_list,genre_count
0,tt0000009,movie,Miss Jerry,Miss Jerry,0,1894.0,\N,45.0,Romance,5.3,232.0,True,False,[Romance],1
1,tt0000147,movie,The Corbett-Fitzsimmons Fight,The Corbett-Fitzsimmons Fight,0,1897.0,\N,100.0,"Documentary,News,Sport",5.3,577.0,True,False,"[Documentary, News, Sport]",3
2,tt0000335,movie,Soldiers of the Cross,Soldiers of the Cross,0,1900.0,\N,40.0,"Biography,Drama",5.5,63.0,True,False,"[Biography, Drama]",2
3,tt0000502,movie,Bohemios,Bohemios,0,1905.0,\N,100.0,\N,3.5,25.0,True,False,[\N],1
4,tt0000574,movie,The Story of the Kelly Gang,The Story of the Kelly Gang,0,1906.0,\N,70.0,"Action,Adventure,Biography",6.0,1036.0,True,False,"[Action, Adventure, Biography]",3
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
722264,tt9916622,movie,Rodolpho Teóphilo - O Legado de um Pioneiro,Rodolpho Teóphilo - O Legado de um Pioneiro,0,2015.0,\N,57.0,Documentary,NaN,NaN,False,True,[Documentary],1
722265,tt9916680,movie,De la ilusión al desconcierto: cine colombiano...,De la ilusión al desconcierto: cine colombiano...,0,2007.0,\N,100.0,Documentary,NaN,NaN,False,True,[Documentary],1
722266,tt9916706,movie,Dankyavar Danka,Dankyavar Danka,0,2013.0,\N,NaN,Comedy,7.7,9.0,True,False,[Comedy],1
722267,tt9916730,movie,6 Gunn,6 Gunn,0,2017.0,\N,116.0,Drama,7.0,13.0,True,False,[Drama],1


## **DIRECTORS & WRITERS**

In [4]:
crew['directors'] = crew['directors'].replace("\\N", pd.NA)
crew['writers']   = crew['writers'].replace("\\N", pd.NA)

crew = crew.dropna(subset=['directors', 'writers'])

crew['directors_nconst'] = crew['directors'].apply(lambda s: [x for x in str(s).split(',') if x])
crew['writers_nconst']   = crew['writers'].apply(lambda s: [x for x in str(s).split(',') if x])

crew = crew[crew['directors_nconst'].apply(len) > 0]
crew = crew[crew['writers_nconst'].apply(len) > 0]

movies = movies.merge(
    crew[['tconst', 'directors_nconst', 'writers_nconst']],
    on='tconst', how='inner'
)

In [5]:
movies

,tconst,titleType,primaryTitle,originalTitle,isAdult,startYear,endYear,runtimeMinutes,genres,averageRating,numVotes,has_rating,is_new,genre_list,genre_count,directors_nconst,writers_nconst
0,tt0000009,movie,Miss Jerry,Miss Jerry,0,1894.0,\N,45.0,Romance,5.3,232.0,True,False,[Romance],1,[nm0085156],[nm0085156]
1,tt0000502,movie,Bohemios,Bohemios,0,1905.0,\N,100.0,\N,3.5,25.0,True,False,[\N],1,[nm0063413],"[nm0063413, nm0657268, nm0675388]"
2,tt0000574,movie,The Story of the Kelly Gang,The Story of the Kelly Gang,0,1906.0,\N,70.0,"Action,Adventure,Biography",6.0,1036.0,True,False,"[Action, Adventure, Biography]",3,[nm0846879],[nm0846879]
3,tt0000591,movie,The Prodigal Son,L'enfant prodigue,0,1907.0,\N,90.0,Drama,5.3,35.0,True,False,[Drama],1,[nm0141150],[nm0141150]
4,tt0000615,movie,Robbery Under Arms,Robbery Under Arms,0,1907.0,\N,NaN,Drama,4.0,31.0,True,False,[Drama],1,[nm0533958],"[nm0092809, nm0533958]"
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
515630,tt9916622,movie,Rodolpho Teóphilo - O Legado de um Pioneiro,Rodolpho Teóphilo - O Legado de um Pioneiro,0,2015.0,\N,57.0,Documentary,NaN,NaN,False,True,[Documentary],1,"[nm9272490, nm9272491]","[nm9272490, nm9272491]"
515631,tt9916680,movie,De la ilusión al desconcierto: cine colombiano...,De la ilusión al desconcierto: cine colombiano...,0,2007.0,\N,100.0,Documentary,NaN,NaN,False,True,[Documentary],1,[nm0652213],"[nm0652213, nm10538576]"
515632,tt9916706,movie,Dankyavar Danka,Dankyavar Danka,0,2013.0,\N,NaN,Comedy,7.7,9.0,True,False,[Comedy],1,[nm7764440],[nm7933903]
515633,tt9916730,movie,6 Gunn,6 Gunn,0,2017.0,\N,116.0,Drama,7.0,13.0,True,False,[Drama],1,[nm10538612],[nm10538612]


## **CASTS**

In [6]:

cast = principals.query("category in ['actor','actress']").copy()
cast = cast[cast['nconst'].notna()]
cast = cast.sort_values(['tconst', 'ordering'])
top_cast = cast.groupby('tconst').head(10)

df_casts_groups = top_cast.groupby('tconst')['nconst'].apply(list).reset_index()
df_casts_groups.rename(columns={'nconst': 'cast_nconst'}, inplace=True)

movies = movies.merge(df_casts_groups, on='tconst', how='inner')

In [7]:
movies

,tconst,titleType,primaryTitle,originalTitle,isAdult,startYear,endYear,runtimeMinutes,genres,averageRating,numVotes,has_rating,is_new,genre_list,genre_count,directors_nconst,writers_nconst,cast_nconst
0,tt0000009,movie,Miss Jerry,Miss Jerry,0,1894.0,\N,45.0,Romance,5.3,232.0,True,False,[Romance],1,[nm0085156],[nm0085156],"[nm0063086, nm0183823, nm1309758]"
1,tt0000502,movie,Bohemios,Bohemios,0,1905.0,\N,100.0,\N,3.5,25.0,True,False,[\N],1,[nm0063413],"[nm0063413, nm0657268, nm0675388]","[nm0215752, nm0252720]"
2,tt0000574,movie,The Story of the Kelly Gang,The Story of the Kelly Gang,0,1906.0,\N,70.0,"Action,Adventure,Biography",6.0,1036.0,True,False,"[Action, Adventure, Biography]",3,[nm0846879],[nm0846879],"[nm0846887, nm0846894, nm1431224, nm3002376, n..."
3,tt0000591,movie,The Prodigal Son,L'enfant prodigue,0,1907.0,\N,90.0,Drama,5.3,35.0,True,False,[Drama],1,[nm0141150],[nm0141150],"[nm0906197, nm0332182, nm1323543, nm1759558]"
4,tt0000615,movie,Robbery Under Arms,Robbery Under Arms,0,1907.0,\N,NaN,Drama,4.0,31.0,True,False,[Drama],1,[nm0533958],"[nm0092809, nm0533958]","[nm3071427, nm0581353, nm0888988, nm0240418, n..."
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
429850,tt9916362,movie,Coven,Akelarre,0,2020.0,\N,92.0,"Drama,History",6.4,6159.0,True,False,"[Drama, History]",2,[nm1893148],"[nm1893148, nm3471432]","[nm3766704, nm0107165, nm0266723, nm10678594, ..."
429851,tt9916538,movie,Kuambil Lagi Hatiku,Kuambil Lagi Hatiku,0,2019.0,\N,123.0,Drama,7.6,12.0,True,False,[Drama],1,[nm4457074],"[nm4843252, nm4900525, nm2679404]","[nm8678236, nm1417182, nm10041459, nm1266058, ..."
429852,tt9916622,movie,Rodolpho Teóphilo - O Legado de um Pioneiro,Rodolpho Teóphilo - O Legado de um Pioneiro,0,2015.0,\N,57.0,Documentary,NaN,NaN,False,True,[Documentary],1,"[nm9272490, nm9272491]","[nm9272490, nm9272491]",[nm9272513]
429853,tt9916706,movie,Dankyavar Danka,Dankyavar Danka,0,2013.0,\N,NaN,Comedy,7.7,9.0,True,False,[Comedy],1,[nm7764440],[nm7933903],"[nm1778107, nm2585097, nm5697682, nm3272130, n..."


## **NAME LOOKUP**

In [8]:
nconst_to_name = dict(zip(name_basic['nconst'], name_basic['primaryName']))
movies['director_names'] = movies['directors_nconst'].apply(lambda lst: [nconst_to_name.get(x) for x in lst] if isinstance(lst, list) else [])
movies['cast_names'] = movies['cast_nconst'].apply(lambda lst: [nconst_to_name.get(x) for x in lst] if isinstance(lst, list) else [])

## **PERSON TRACK-RECORDS**

In [9]:
person_movies = principals[['nconst','tconst']].copy()

# Add directors
dir_expl = crew[['tconst','directors_nconst']].explode('directors_nconst').dropna()
dir_expl = dir_expl.rename(columns={'directors_nconst':'nconst'})

# Combine
person_movies = pd.concat([person_movies, dir_expl[['nconst','tconst']]], ignore_index=True)

# Merge ratings to each person's movies
person_movies = person_movies.merge(
    ratings[['tconst','averageRating','numVotes']],
    on='tconst', how='left'
)

In [10]:
# Aggregate stats per person
person_stats = person_movies.groupby('nconst').agg(
    person_mean_rating=('averageRating','mean'),
    person_median_rating=('averageRating','median'),
    person_count_films=('tconst','nunique'),
    person_mean_votes=('numVotes','mean')
).reset_index()

In [11]:
# Build lookup dicts
mean_map = dict(zip(person_stats['nconst'], person_stats['person_mean_rating']))
count_map = dict(zip(person_stats['nconst'], person_stats['person_count_films']))

def agg_person_metric(nconst_list, metric_map, func=np.mean):
    if not isinstance(nconst_list, list):
        return np.nan
    vals = [metric_map.get(x) for x in nconst_list if metric_map.get(x) is not None]
    return func(vals) if len(vals) else np.nan

# director stats
movies['director_mean_rating'] = movies['directors_nconst'].apply(lambda L: agg_person_metric(L, mean_map))
movies['director_total_films'] = movies['directors_nconst'].apply(lambda L: agg_person_metric(L, count_map, func=np.sum))

# cast stats
movies['cast_mean_rating'] = movies['cast_nconst'].apply(lambda L: agg_person_metric(L, mean_map))
movies['cast_total_films'] = movies['cast_nconst'].apply(lambda L: agg_person_metric(L, count_map, func=np.sum))

## **NaN VALUES PROCESSING**

In [13]:

# === TÍNH GLOBAL MEAN RATING ===
director_mean_rating = movies['director_mean_rating'].mean()
cast_mean_rating = movies['cast_mean_rating'].mean()

# Fill mean rating cho director và cast
movies['director_mean_rating'] = movies['director_mean_rating'].fillna(director_mean_rating)
movies['cast_mean_rating']     = movies['cast_mean_rating'].fillna(cast_mean_rating)

# các director/cast chưa đóng phim nào khác
movies['director_total_films'] = movies['director_total_films'].fillna(0)
movies['cast_total_films']     = movies['cast_total_films'].fillna(0)

## **LABEL**

In [14]:
movies['is_success'] = (
    (movies['averageRating'] >= 7.0) &
    (movies['numVotes'] >= 30000)
).astype(int)

movies['is_risky'] = 1 - movies['is_success']

In [15]:
one_big_table = movies.copy()

In [16]:
one_big_table

,tconst,titleType,primaryTitle,originalTitle,isAdult,startYear,endYear,runtimeMinutes,genres,averageRating,...,writers_nconst,cast_nconst,director_names,cast_names,director_mean_rating,director_total_films,cast_mean_rating,cast_total_films,is_success,is_risky
0,tt0000009,movie,Miss Jerry,Miss Jerry,0,1894.0,\N,45.0,Romance,5.3,...,[nm0085156],"[nm0063086, nm0183823, nm1309758]",[Alexander Black],"[Blanche Bayliss, William Courtenay, Chauncey ...",5.300000,1,5.477778,22,0,1
1,tt0000502,movie,Bohemios,Bohemios,0,1905.0,\N,100.0,\N,3.5,...,"[nm0063413, nm0657268, nm0675388]","[nm0215752, nm0252720]",[Ricardo de Baños],"[Antonio del Pozo, El Mochuelo]",4.386486,108,3.500000,2,0,1
2,tt0000574,movie,The Story of the Kelly Gang,The Story of the Kelly Gang,0,1906.0,\N,70.0,"Action,Adventure,Biography",6.0,...,[nm0846879],"[nm0846887, nm0846894, nm1431224, nm3002376, n...",[Charles Tait],"[Elizabeth Tait, John Tait, Nicholas Brierley,...",6.000000,1,5.634333,42,0,1
3,tt0000591,movie,The Prodigal Son,L'enfant prodigue,0,1907.0,\N,90.0,Drama,5.3,...,[nm0141150],"[nm0906197, nm0332182, nm1323543, nm1759558]",[Michel Carré],"[Georges Wague, Henri Gouget, Christiane Mande...",5.593750,64,5.532023,151,0,1
4,tt0000615,movie,Robbery Under Arms,Robbery Under Arms,0,1907.0,\N,NaN,Drama,4.0,...,"[nm0092809, nm0533958]","[nm3071427, nm0581353, nm0888988, nm0240418, n...",[Charles MacMahon],"[Jim Gerald, George Merriman, Lance Vane, Will...",4.000000,2,4.200000,13,0,1
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
429850,tt9916362,movie,Coven,Akelarre,0,2020.0,\N,92.0,"Drama,History",6.4,...,"[nm1893148, nm3471432]","[nm3766704, nm0107165, nm0266723, nm10678594, ...",[Pablo Agüero],"[Amaia Aberasturi, Alex Brendemühl, Daniel Fan...",5.796552,17,6.615587,2374,0,1
429851,tt9916538,movie,Kuambil Lagi Hatiku,Kuambil Lagi Hatiku,0,2019.0,\N,123.0,Drama,7.6,...,"[nm4843252, nm4900525, nm2679404]","[nm8678236, nm1417182, nm10041459, nm1266058, ...",[Azhar Kinoi Lubis],"[Lala Karmela, Cut Mini Theo, Sahil Shah, Ria ...",6.208889,55,6.634945,335,0,1
429852,tt9916622,movie,Rodolpho Teóphilo - O Legado de um Pioneiro,Rodolpho Teóphilo - O Legado de um Pioneiro,0,2015.0,\N,57.0,Documentary,NaN,...,"[nm9272490, nm9272491]",[nm9272513],"[Angela Gurgel, Ana Célia de Oliveira]",[Oldair Soares Ammom],6.233025,113,6.355243,4,0,1
429853,tt9916706,movie,Dankyavar Danka,Dankyavar Danka,0,2013.0,\N,NaN,Comedy,7.7,...,[nm7933903],"[nm1778107, nm2585097, nm5697682, nm3272130, n...",[Kanchan Nayak],"[Makarand Anaspure, Anvay Bendre, Prakash Dhot...",7.800000,4,7.166570,4740,0,1
